In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
%matplotlib inline

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


In [2]:
# Load the news dataset
print("Loading news data...")
news_df = pd.read_csv('../data/raw/raw_analyst_ratings.csv', index_col=0)
print(f"✅ News data loaded! Shape: {news_df.shape}")

# Load the stock data with indicators we saved in Task 2
print("\nLoading stock data...")
stock_df = pd.read_csv('../data/raw/aapl_with_indicators.csv', index_col=0)
print(f"✅ Stock data loaded! Shape: {stock_df.shape}")

# Preview both
print("\n=== NEWS DATA SAMPLE ===")
print(news_df.head(3))
print("\n=== STOCK DATA SAMPLE ===")
print(stock_df.head(3))

Loading news data...
✅ News data loaded! Shape: (1407328, 5)

Loading stock data...
✅ Stock data loaded! Shape: (3524, 14)

=== NEWS DATA SAMPLE ===
                                     headline  \
0     Stocks That Hit 52-Week Highs On Friday   
1  Stocks That Hit 52-Week Highs On Wednesday   
2               71 Biggest Movers From Friday   

                                                 url          publisher  \
0  https://www.benzinga.com/news/20/06/16190091/s...  Benzinga Insights   
1  https://www.benzinga.com/news/20/06/16170189/s...  Benzinga Insights   
2  https://www.benzinga.com/news/20/05/16103463/7...         Lisa Levin   

                        date stock  
0  2020-06-05 10:30:54-04:00     A  
1  2020-06-03 10:45:20-04:00     A  
2  2020-05-26 04:30:07-04:00     A  

=== STOCK DATA SAMPLE ===
                        Close               High                Low  \
Price                                                                 
Ticker                   AAPL       

In [3]:
import yfinance as yf

print("Re-downloading clean AAPL stock data...")
aapl_clean = yf.download("AAPL", start="2020-01-01", end="2024-01-01", 
                          auto_adjust=True, progress=False)

# Flatten columns if multi-level
if isinstance(aapl_clean.columns, pd.MultiIndex):
    aapl_clean.columns = aapl_clean.columns.get_level_values(0)

# Keep only what we need
aapl_clean = aapl_clean[['Open', 'High', 'Low', 'Close', 'Volume']]
aapl_clean = aapl_clean.dropna()

# Compute daily return
aapl_clean['Daily_Return'] = aapl_clean['Close'].pct_change() * 100
aapl_clean = aapl_clean.dropna()

# Make sure index is datetime
aapl_clean.index = pd.to_datetime(aapl_clean.index)
# Remove timezone info from stock data
aapl_clean.index = aapl_clean.index.tz_localize(None)

print(f"✅ Clean stock data shape: {aapl_clean.shape}")
print(f"Date range: {aapl_clean.index[0].date()} to {aapl_clean.index[-1].date()}")
print(aapl_clean.head(3))

# ============================================
# CLEAN NEWS DATA
# ============================================
# Filter only AAPL news
aapl_news = news_df[news_df['stock'] == 'AAPL'].copy()
print(f"\n✅ AAPL news articles: {len(aapl_news)}")

# Parse dates
aapl_news['date'] = pd.to_datetime(aapl_news['date'], utc=True, errors='coerce')
aapl_news = aapl_news.dropna(subset=['date'])

# Remove timezone and keep only date part
aapl_news['date'] = aapl_news['date'].dt.tz_localize(None)
aapl_news['trading_date'] = aapl_news['date'].dt.normalize()

print(f"Date range: {aapl_news['trading_date'].min().date()} to {aapl_news['trading_date'].max().date()}")
print(aapl_news[['headline', 'trading_date']].head(3))

Re-downloading clean AAPL stock data...
✅ Clean stock data shape: (1005, 6)
Date range: 2020-01-03 to 2023-12-29
Price            Open       High        Low      Close     Volume  \
Date                                                                
2020-01-03  71.629145  72.455958  71.472462  71.696640  146322800   
2020-01-06  70.819231  72.306529  70.568532  72.267960  118387200   
2020-01-07  72.277586  72.533103  71.708703  71.928062  108872000   

Price       Daily_Return  
Date                      
2020-01-03     -0.972193  
2020-01-06      0.796857  
2020-01-07     -0.470329  

✅ AAPL news articles: 441
Date range: 2020-06-09 to 2020-06-10
                                               headline trading_date
7120  Tech Stocks And FAANGS Strong Again To Start D...   2020-06-10
7121      10 Biggest Price Target Changes For Wednesday   2020-06-10
7122  Benzinga Pro's Top 5 Stocks To Watch For Wed.,...   2020-06-10


In [4]:
print("=== CHECKING APPLE-RELATED SYMBOLS IN NEWS DATA ===")

# Check what symbols exist that might be Apple
apple_symbols = ['AAPL', 'APPLE', 'Apple']
for sym in apple_symbols:
    count = len(news_df[news_df['stock'] == sym])
    print(f"'{sym}': {count} articles")

# Check top symbols overall
print("\n=== TOP 20 STOCK SYMBOLS IN NEWS DATA ===")
print(news_df['stock'].value_counts().head(20))

=== CHECKING APPLE-RELATED SYMBOLS IN NEWS DATA ===
'AAPL': 441 articles
'APPLE': 0 articles
'Apple': 0 articles

=== TOP 20 STOCK SYMBOLS IN NEWS DATA ===
stock
MRK     3333
MS      3238
NVDA    3146
MU      3142
QQQ     3106
NFLX    3028
M       3025
EBAY    3018
GILD    2968
VZ      2966
QCOM    2941
JNJ     2928
DAL     2926
BABA    2858
KO      2797
AA      2739
EWU     2702
ORCL    2701
FDX     2629
HD      2612
Name: count, dtype: int64


In [5]:
# ============================================
# USE NVDA - Best covered tech stock
# ============================================

# Download NVDA stock data
print("Downloading NVDA stock data...")
nvda_clean = yf.download("NVDA", start="2010-01-01", end="2024-01-01",
                          auto_adjust=True, progress=False)

# Flatten columns if multi-level
if isinstance(nvda_clean.columns, pd.MultiIndex):
    nvda_clean.columns = nvda_clean.columns.get_level_values(0)

# Keep only what we need
nvda_clean = nvda_clean[['Open', 'High', 'Low', 'Close', 'Volume']]
nvda_clean = nvda_clean.dropna()

# Compute daily return
nvda_clean['Daily_Return'] = nvda_clean['Close'].pct_change() * 100
nvda_clean = nvda_clean.dropna()

# Make sure index is datetime without timezone
nvda_clean.index = pd.to_datetime(nvda_clean.index)
nvda_clean.index = nvda_clean.index.tz_localize(None)

print(f"✅ NVDA stock data shape: {nvda_clean.shape}")
print(f"Date range: {nvda_clean.index[0].date()} to {nvda_clean.index[-1].date()}")

# ============================================
# FILTER NVDA NEWS
# ============================================
nvda_news = news_df[news_df['stock'] == 'NVDA'].copy()
print(f"\n✅ NVDA news articles: {len(nvda_news)}")

# Parse dates
nvda_news['date'] = pd.to_datetime(nvda_news['date'], utc=True, errors='coerce')
nvda_news = nvda_news.dropna(subset=['date'])

# Remove timezone
nvda_news['date'] = nvda_news['date'].dt.tz_localize(None)
nvda_news['trading_date'] = nvda_news['date'].dt.normalize()

print(f"News date range: {nvda_news['trading_date'].min().date()} to {nvda_news['trading_date'].max().date()}")
print(f"\nSample headlines:")
print(nvda_news['headline'].head(5).values)

✅ NVDA stock data shape: (3521, 6)
Date range: 2010-01-05 to 2023-12-29

✅ NVDA news articles: 3146
News date range: 2020-05-31 to 2020-06-10

Sample headlines:
<StringArray>
['Shares of several technology companies are trading higher on continued volatility despite market weakness. The sector sold off recently as other sectors gained amid US economic reopening and appears to be rebounding.',
                                                                                                                                                                                      'Afternoon Market Stats in 5 Minutes',
                                                                                                                                                                                        'Morning Market Stats in 5 Minutes',
                                              'Shares of several technology companies are trading higher despite market weakness. The sector recently experienced 

In [7]:
# ============================================
# FIX: Re-download NVDA stock data to match
# the news date range properly
# ============================================

# First check the full news date range
nvda_news_all = news_df[news_df['stock'] == 'NVDA'].copy()
nvda_news_all['date'] = pd.to_datetime(nvda_news_all['date'], utc=True, errors='coerce')
nvda_news_all = nvda_news_all.dropna(subset=['date'])
nvda_news_all['date'] = nvda_news_all['date'].dt.tz_localize(None)
nvda_news_all['trading_date'] = nvda_news_all['date'].dt.normalize()

print("=== NEWS DATE RANGE ANALYSIS ===")
print(f"Earliest news: {nvda_news_all['trading_date'].min()}")
print(f"Latest news: {nvda_news_all['trading_date'].max()}")
print(f"\nNews articles per year:")
print(nvda_news_all['trading_date'].dt.year.value_counts().sort_index())

print(f"\nNews articles per month:")
print(nvda_news_all['trading_date'].dt.to_period('M').value_counts().sort_index())

=== NEWS DATE RANGE ANALYSIS ===
Earliest news: 2020-05-31 00:00:00
Latest news: 2020-06-10 00:00:00

News articles per year:
trading_date
2020    10
Name: count, dtype: int64

News articles per month:
trading_date
2020-05    1
2020-06    9
Freq: M, Name: count, dtype: int64


In [8]:
# ============================================
# FIND BEST STOCK WITH MOST DATE COVERAGE
# ============================================

print("Analyzing date coverage for top stocks...\n")

top_stocks = ['MRK', 'MS', 'NVDA', 'MU', 'NFLX', 'GILD', 'QCOM', 'BABA', 'KO', 'ORCL']

for symbol in top_stocks:
    temp = news_df[news_df['stock'] == symbol].copy()
    temp['date'] = pd.to_datetime(temp['date'], utc=True, errors='coerce')
    temp = temp.dropna(subset=['date'])
    temp['date'] = temp['date'].dt.tz_localize(None)
    
    if len(temp) > 0:
        min_date = temp['date'].min().date()
        max_date = temp['date'].max().date()
        days_covered = (temp['date'].max() - temp['date'].min()).days
        print(f"{symbol}: {len(temp)} articles | {min_date} to {max_date} | {days_covered} days covered")

Analyzing date coverage for top stocks...

MRK: 10 articles | 2020-06-03 to 2020-06-11 | 7 days covered
MS: 10 articles | 2020-06-05 to 2020-06-11 | 6 days covered
NVDA: 10 articles | 2020-05-31 to 2020-06-10 | 10 days covered
MU: 10 articles | 2020-05-28 to 2020-06-10 | 12 days covered
NFLX: 10 articles | 2020-06-02 to 2020-06-10 | 8 days covered
GILD: 10 articles | 2020-06-08 to 2020-06-10 | 2 days covered
QCOM: 10 articles | 2020-05-27 to 2020-06-08 | 12 days covered
BABA: 10 articles | 2020-06-01 to 2020-06-10 | 9 days covered
KO: 10 articles | 2020-04-22 to 2020-06-11 | 50 days covered
ORCL: 10 articles | 2020-04-17 to 2020-06-11 | 55 days covered


In [9]:
# ============================================
# STRATEGY: Use ALL stocks combined
# This gives us maximum data for correlation
# ============================================

print("Analyzing full dataset date coverage...")

# Parse all dates in news dataset
news_all = news_df.copy()
news_all['date'] = pd.to_datetime(news_all['date'], utc=True, errors='coerce')
news_all = news_all.dropna(subset=['date'])
news_all['date'] = news_all['date'].dt.tz_localize(None)
news_all['trading_date'] = news_all['date'].dt.normalize()

print(f"Total articles with valid dates: {len(news_all)}")
print(f"Overall date range: {news_all['trading_date'].min().date()} to {news_all['trading_date'].max().date()}")
print(f"Unique stocks: {news_all['stock'].nunique()}")
print(f"\nArticles per year:")
print(news_all['trading_date'].dt.year.value_counts().sort_index())

Analyzing full dataset date coverage...
Total articles with valid dates: 55987
Overall date range: 2011-04-28 to 2020-06-11
Unique stocks: 6204

Articles per year:
trading_date
2011      760
2012     1181
2013     1246
2014     1189
2015     3695
2016     4223
2017     3581
2018     5395
2019     6325
2020    28392
Name: count, dtype: int64


In [10]:
# ============================================
# SENTIMENT ANALYSIS USING VADER
# We use VADER because:
# - Designed for financial/social media text
# - Works well with short headlines
# - No training required
# - Returns compound score (-1 to +1)
# ============================================

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment_score(headline):
    """Returns compound sentiment score between -1 and 1"""
    try:
        score = analyzer.polarity_scores(str(headline))
        return score['compound']
    except:
        return 0.0

print("Running sentiment analysis on all headlines...")
print("This may take a moment with 55,987 articles...\n")

# Apply sentiment scoring
news_all['sentiment_score'] = news_all['headline'].apply(get_sentiment_score)

print("✅ Sentiment analysis complete!")
print(f"\nSentiment score statistics:")
print(news_all['sentiment_score'].describe())

print(f"\nSample scores:")
print(news_all[['headline', 'sentiment_score']].head(10))

Running sentiment analysis on all headlines...
This may take a moment with 55,987 articles...

✅ Sentiment analysis complete!

Sentiment score statistics:
count    55987.000000
mean         0.066879
std          0.313680
min         -0.938200
25%          0.000000
50%          0.000000
75%          0.202300
max          0.966600
Name: sentiment_score, dtype: float64

Sample scores:
                                            headline  sentiment_score
0            Stocks That Hit 52-Week Highs On Friday            0.000
1         Stocks That Hit 52-Week Highs On Wednesday            0.000
2                      71 Biggest Movers From Friday            0.000
3       46 Stocks Moving In Friday's Mid-Day Session            0.000
4  B of A Securities Maintains Neutral on Agilent...            0.296
5  CFRA Maintains Hold on Agilent Technologies, L...           -0.128
6  UBS Maintains Neutral on Agilent Technologies,...            0.000
7  Agilent Technologies shares are trading higher...   

In [11]:
# ============================================
# CLASSIFY SENTIMENT AND AGGREGATE BY DAY
# ============================================

# Classify each article as positive, neutral, or negative
def classify_sentiment(score):
    if score > 0.05:
        return 'positive'
    elif score < -0.05:
        return 'negative'
    else:
        return 'neutral'

news_all['sentiment_label'] = news_all['sentiment_score'].apply(classify_sentiment)

# Show sentiment distribution
print("=== SENTIMENT DISTRIBUTION ===")
sentiment_counts = news_all['sentiment_label'].value_counts()
print(sentiment_counts)
print(f"\nPercentages:")
print((sentiment_counts / len(news_all) * 100).round(2))

# Aggregate daily sentiment per stock
# If multiple articles per stock per day → take average
print("\nAggregating daily sentiment scores per stock...")
daily_sentiment = news_all.groupby(['stock', 'trading_date']).agg(
    avg_sentiment=('sentiment_score', 'mean'),
    article_count=('headline', 'count'),
    sentiment_label=('sentiment_label', lambda x: x.mode()[0])
).reset_index()

print(f"✅ Daily sentiment aggregated!")
print(f"Shape: {daily_sentiment.shape}")
print(f"\nSample:")
print(daily_sentiment.head(10))

=== SENTIMENT DISTRIBUTION ===
sentiment_label
neutral     26364
positive    16423
negative    13200
Name: count, dtype: int64

Percentages:
sentiment_label
neutral     47.09
positive    29.33
negative    23.58
Name: count, dtype: float64

Aggregating daily sentiment scores per stock...
✅ Daily sentiment aggregated!
Shape: (44204, 5)

Sample:
  stock trading_date  avg_sentiment  article_count sentiment_label
0     A   2020-05-22         0.0480              7         neutral
1     A   2020-05-26         0.0000              1         neutral
2     A   2020-06-03         0.0000              1         neutral
3     A   2020-06-05         0.0000              1         neutral
4    AA   2020-05-18         0.8519              1        positive
5    AA   2020-05-26        -0.1280              1        negative
6    AA   2020-05-27         0.4455              2         neutral
7    AA   2020-06-03         0.8442              1        positive
8    AA   2020-06-04         0.8442              1  